# El objetivo de este notebook

Agruparemos reservas por fecha de estadía (no solo `arrival_date`, sino cada noche ocupada dentro del rango de estancia), contaremos habitaciones ocupadas por día y por hotel, y tomaremos el día de mayor ocupación como proxy de "capacidad total" de ese hotel.

---

In [1]:
import pandas as pd
from sqlalchemy import create_engine
import os
import sys
sys.path.append(os.path.abspath(os.path.join("..")))
from src.load import load_to_sqlite

In [2]:
# Extraemos nuestro historico de reservas de la tabla 'clean_bookings'en la base de datos '\data\hotel_data.db'

def extract_from_sqlite(db_path: str = os.path.join("data", "hotel_data.db"))  -> pd.DataFrame:
    """
    Conecta a la base de datos SQLite y extrae los datos de la tabla 'clean_bookings'.

    Args:
        db_path (str): Ruta al archivo de la base de datos SQLite.

    Returns:
        pd.DataFrame: DataFrame que contiene los datos extraídos.
    """
    if not os.path.exists(db_path):
        raise FileNotFoundError(f"❌ Error: No se encontró el archivo de base de datos en {db_path}. Asegúrate de que la base de datos exista.")
    
    print(f"⏳ Conectando a la base de datos en {db_path}...")

    # Crear la conexión a la base de datos SQLite
    engine = create_engine(f'sqlite:///{db_path}')

    # Consulta SQL para extraer los datos de la tabla 'clean_bookings'
    query = "SELECT * FROM clean_bookings"

    print("⏳ Ejecutando la consulta SQL para extraer los datos...")

    # Leer los datos de la tabla especificada en un DataFrame
    df = pd.read_sql_query(query, con=engine)

    print(f"✅ Extracción exitosa. Filas extraídas: {df.shape[0]}, Columnas: {df.shape[1]}")
    
    return df

In [3]:
df = extract_from_sqlite()

⏳ Conectando a la base de datos en data\hotel_data.db...
⏳ Ejecutando la consulta SQL para extraer los datos...
✅ Extracción exitosa. Filas extraídas: 119317, Columnas: 43


## 1. Expansión a nivel noche-reserva

Nuestro dataset tiene una fila por reserva, no por noche ocupada. Una reserva con `arrival_date` = 5 julio y `total_nights` = 4 ocupa una habitación el 5, 6, 7 y 8 de julio. Si agrupamos por `arrival_date` tal cual, solo contamos cuántas reservas llegaron ese día, no cuántas habitaciones estaban físicamente ocupadas ese día (que incluye llegadas de días anteriores que siguen hospedadas). Para simular capacidad necesitamos lo segundo.

Usaremos columnas que ya creamos en `transform.py`:

In [4]:
def expand_to_nightly_occupancy(df: pd.DataFrame) -> pd.DataFrame:
    """
    Expande cada reserva (no cancelada) en una fila por cada noche ocupada.
    Necesario para calcular ocupación real diaria y simular capacidad de inventario.
    """
    df = df[df['is_canceled'] == 0].copy()  # Descartamos las reservas canceladas

    # Reconstruimos la fecha real de llegada
    df['arrival_date'] = pd.to_datetime(
        df['arrival_date_year'].astype(str) + '-' +
        df['arrival_date_month'].astype(str) + '-' +
        df['arrival_date_day_of_month'].astype(str),
        format='%Y-%B-%d'
    )

    rows = []
    for _, row in df.iterrows():
        for n in range(row['total_nights']):
            rows.append({
                'hotel': row['hotel'],
                'room_type': row['assigned_room_type'],  # Usamos como referencia la habitación que REALMENTE se ocupa, no la que se reserva
                'stay_date': row['arrival_date'] + pd.Timedelta(days=n)
            })

    return pd.DataFrame(rows)

In [5]:
df_nightly = expand_to_nightly_occupancy(df)

In [6]:
display(df_nightly.head())

,hotel,room_type,stay_date
0,Resort Hotel,C,2015-07-01
1,Resort Hotel,A,2015-07-01
2,Resort Hotel,A,2015-07-01
3,Resort Hotel,A,2015-07-02
4,Resort Hotel,A,2015-07-01


## 2. Calculando el máximo de ocupacion por hotel y tipología

In [7]:
def calculate_simulated_capacity(df_nightly: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula la capacidad simulada por hotel + tipo de habitación,
    usando el máximo histórico de ocupación diaria observada.
    Incluye sanity check: cuántas veces se alcanzó ese máximo.
    """
    ocupacion_diaria = (
        df_nightly.groupby(['hotel', 'room_type', 'stay_date'])
        .size()
        .reset_index(name='rooms_occupied')
    )

    capacidad = (
        ocupacion_diaria.groupby(['hotel', 'room_type'])['rooms_occupied']
        .max()
        .reset_index(name='capacidad_simulada')
    )

    return ocupacion_diaria, capacidad

In [8]:
ocupacion_diaria, inventario = calculate_simulated_capacity(df_nightly)

In [12]:
display(ocupacion_diaria.head())

,hotel,room_type,stay_date,rooms_occupied
0,City Hotel,A,2015-07-01,65
1,City Hotel,A,2015-07-02,66
2,City Hotel,A,2015-07-03,13
3,City Hotel,A,2015-07-04,22
4,City Hotel,A,2015-07-05,12


In [9]:
display(inventario)

,hotel,room_type,capacidad_simulada
0,City Hotel,A,146
1,City Hotel,B,15
2,City Hotel,C,3
3,City Hotel,D,57
4,City Hotel,E,10
5,City Hotel,F,8
6,City Hotel,G,5
7,City Hotel,K,8
8,Resort Hotel,A,75
9,Resort Hotel,B,2


In [10]:
# Ejecutamos la carga para el inventario

load_to_sqlite(inventario, "inventory", "replace")

💾 Iniciando proceso de carga de datos...
    ├─ Conectando a la base de datos en 'data/hotel_data.db'...
    ├─ Insertando 17 registros en la tabla 'inventory' (Estrategia: replace)...
    └─ Verificación de carga exitosa: 17 filas en la tabla 'inventory'.
✅ Proceso de carga completado con éxito.



In [11]:
# Ejecutamos la carga para la ocupación diaria
load_to_sqlite(ocupacion_diaria, "occupancy", "replace")

💾 Iniciando proceso de carga de datos...
    ├─ Conectando a la base de datos en 'data/hotel_data.db'...
    ├─ Insertando 11,504 registros en la tabla 'occupancy' (Estrategia: replace)...
    └─ Verificación de carga exitosa: 11,504 filas en la tabla 'occupancy'.
✅ Proceso de carga completado con éxito.



In [13]:
# Ejecutamos la carga para los datos crudos de ocupacion por noche

load_to_sqlite(df_nightly, "occupancy_raw", "replace")

💾 Iniciando proceso de carga de datos...
    ├─ Conectando a la base de datos en 'data/hotel_data.db'...
    ├─ Insertando 254,932 registros en la tabla 'occupancy_raw' (Estrategia: replace)...
    └─ Verificación de carga exitosa: 254,932 filas en la tabla 'occupancy_raw'.
✅ Proceso de carga completado con éxito.

